# Pandas

Практические ячейки к слайдам презентации. Заголовки секций совпадают с заголовками слайдов.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)



## Series и DataFrame `(example)`


Базовые структуры Pandas.


In [ ]:
import pandas as pd

s = pd.Series([10, 20, 15], index=["a", "b", "c"])
df = pd.DataFrame({"Age": [22, 38, 26], "Fare": [7.25, 71.28, 7.92], "Sex": ["male", "female", "female"]})

print("Series:\n", s)
print("\nDataFrame shape:", df.shape)
print(df.dtypes)



## Чтение данных `(example)`


In [ ]:
from sklearn.datasets import fetch_openml

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"]
df["Embarked"] = df["embarked"]

print(df.shape)
df.head()



## Первичный осмотр таблицы `(example)`


In [ ]:
df.info()
display(df.describe())
print("Sex counts:\n", df["Sex"].value_counts())



## Индексы строк и столбцов `(example)`


In [ ]:
df_idx = df.set_index("name")
print(df_idx.index[:3])
df_back = df_idx.reset_index()
print("После reset_index:", df_back.columns[:3].tolist())



## loc и iloc: две системы адресации `(example)`


In [ ]:
sample = df.head(8)
print("loc по меткам 3:5, столбцы Age/Fare:")
display(sample.loc[3:5, ["Age", "Fare"]])
print("iloc по позициям 0:3, столбцы 1:3:")
display(sample.iloc[0:3, 1:3])



## Фильтрация и булевы маски `(example)`


In [ ]:
mask = (df["Age"] > 30) & (df["Pclass"] == 1)
filtered = df.loc[mask, ["Sex", "Age", "Pclass", "Survived"]]
print(f"Строк по маске: {len(filtered)}")
filtered.head()



## groupby и агрегации `(example)`


In [ ]:
by_class = df.groupby("Pclass").agg({"Fare": "mean", "Age": "median", "Survived": "mean"})
by_class.round(2)



## merge и join `(example)`


In [ ]:
ports = pd.DataFrame({"Embarked": ["S", "C", "Q"], "port_name": ["Southampton", "Cherbourg", "Queenstown"]})
merged = pd.merge(df[["Embarked", "Fare"]].drop_duplicates("Embarked"), ports, on="Embarked", how="left")
merged



## Пропуски: isna, fillna, dropna `(experiment)`


In [ ]:
print("Пропуски Age:", df["Age"].isna().sum())
df_filled = df.copy()
df_filled["Age"] = df_filled["Age"].fillna(df_filled["Age"].median())
print("После fillna median:", df_filled["Age"].isna().sum())



## Типы данных `(example)`


In [ ]:
df_cat = df.copy()
df_cat["Sex"] = df_cat["Sex"].astype("category")
print(df_cat["Sex"].dtype)
print("Память object vs category:", df["Sex"].memory_usage(deep=True), df_cat["Sex"].memory_usage(deep=True))



## apply и transform `(example)`


In [ ]:
emb_map = {"S": 0, "C": 1, "Q": 2}
df["Embarked_code"] = df["Embarked"].map(emb_map)
df["Fare_z"] = df.groupby("Pclass")["Fare"].transform(lambda x: (x - x.mean()) / x.std())
df[["Embarked", "Embarked_code", "Pclass", "Fare_z"]].head()



## pivot и pivot_table `(example)`


In [ ]:
pt = df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean")
pt.round(2)



## Базовая производительность `(experiment)`


In [ ]:
import time
s = pd.Series(range(200_000))
t0 = time.perf_counter(); _ = s.apply(lambda x: x * 2); t1 = time.perf_counter()
t2 = time.perf_counter(); _ = s * 2; t3 = time.perf_counter()
print(f"apply: {t1-t0:.3f}s, vectorized: {t3-t2:.3f}s")



## Pandas и sklearn Pipeline `(example)`


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = ["Pclass", "Age", "Fare"]
X = df[features]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
pipe.fit(X_train, y_train)
print(f"Test accuracy: {pipe.score(X_test, y_test):.3f}")

